In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import os
import shutil
import pandas as pd

# Load TDR numbers
df = pd.read_excel("MP_pipe_data.xlsx")
tdr_list = df["TDR"].astype(str).tolist()

# Setup directories
base_download_dir = os.path.join(os.getcwd(), "BOQ_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

# Login manually
driver.get("https://www.tenderdetail.com/Account/LogOn")
input("🔐 Log in manually, then press ENTER to continue...")

summary = []

# Process each TDR
for tdr in tdr_list:
    print(f"🔎 Searching for TDR: {tdr}")
    tdr_dir = os.path.join(base_download_dir, tdr)
    os.makedirs(tdr_dir, exist_ok=True)

    driver.get(f"https://www.tenderdetail.com/registeruser/searchtenders?ServiceType=1&nm={tdr}")
    time.sleep(3)

    try:
        tender_brief = driver.find_element(By.CSS_SELECTOR, "#tenderbreif > span").text
        if tdr not in tender_brief:
            summary.append({"TDR": tdr, "BOQ": "", "NIT": "", "BDE": ""})
            continue
    except:
        summary.append({"TDR": tdr, "BOQ": "", "NIT": "", "BDE": ""})
        continue

    try:
        view_notice = driver.find_element(By.CSS_SELECTOR, ".viewnotice")
        driver.execute_script("arguments[0].click();", view_notice)
        time.sleep(3)
    except:
        summary.append({"TDR": tdr, "BOQ": "", "NIT": "", "BDE": ""})
        continue

    driver.switch_to.window(driver.window_handles[-1])
    time.sleep(3)

    # Track downloaded filenames
    boq_file = ""
    nit_file = ""
    bde_file = ""

    try:
        all_links = driver.find_elements(By.TAG_NAME, "a")
        for i in range(len(all_links)):
            all_links = driver.find_elements(By.TAG_NAME, "a")
            link = all_links[i]
            href = link.get_attribute("href")
            text = link.text.lower()

            # Extract filename from href
            if not href or not any(ext in href.lower() for ext in [".pdf", ".xls", ".xlsx"]):
                continue

            filename_param = ""
            if "FileName=" in href:
                filename_param = href.split("FileName=")[-1]
            else:
                filename_param = os.path.basename(href.split("?")[0])

            file_key = ""
            if "boq" in filename_param.lower():
                file_key = "BOQ"
                boq_file = filename_param
            elif "nit" in filename_param.lower():
                file_key = "NIT"
                nit_file = filename_param
            elif "bde" in filename_param.lower():
                file_key = "BDE"
                bde_file = filename_param
            else:
                continue  # skip unrelated files

            print(f"📥 Downloading {file_key}: {filename_param}")
            driver.get(href)
            time.sleep(5)

    except Exception as e:
        print(f"❌ Error for TDR {tdr}: {e}")

    # Move files to TDR folder
    try:
        for f in os.listdir(base_download_dir):
            full_path = os.path.join(base_download_dir, f)
            if os.path.isfile(full_path) and any(f.endswith(ext) for ext in [".pdf", ".xls", ".xlsx"]):
                shutil.move(full_path, os.path.join(tdr_dir, f))
    except Exception as move_err:
        print(f"⚠️ Move error: {move_err}")

    # Append result
    summary.append({
        "TDR": tdr,
        "BOQ": boq_file,
        "NIT": nit_file,
        "BDE": bde_file
    })

    # Close and switch tab
    if len(driver.window_handles) > 1:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
    time.sleep(2)

# Save summary
summary_df = pd.DataFrame(summary)
summary_df.to_excel(os.path.join(base_download_dir, "summary.xlsx"), index=False)

print("✅ Done! Summary saved.")
driver.quit()
